# Черновики аудиофрагментов для сайта
1. Загрузи `borch_20min.zip` в Google Drive → папка `avar_site`.
2. Среда выполнения → Сменить среду → **T4 GPU**.
3. Запусти ячейку. Результат: `avar_site/borch_20min_drafts.zip`.

In [ ]:
# Черновики для сайта: 20 минут «Борча» → исходная модель MMS (ava) без дообучения → metadata.csv
# Положи borch_20min.zip в Google Drive: MyDrive/avar_site/borch_20min.zip. Среда: GPU (T4). Время ≈ 5–10 мин.
import os, re, io, zipfile, unicodedata
from google.colab import drive
drive.mount('/content/drive')
SRC = '/content/drive/MyDrive/avar_site/borch_20min.zip'
OUT = '/content/drive/MyDrive/avar_site/borch_20min_drafts.zip'
assert os.path.exists(SRC), 'Нет файла ' + SRC + ' — загрузи borch_20min.zip в папку avar_site на Google Drive'

import torch, numpy as np, soundfile as sf
from transformers import Wav2Vec2ForCTC, AutoProcessor
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
MID = 'facebook/mms-1b-all'
proc = AutoProcessor.from_pretrained(MID, target_lang='ava')
model = Wav2Vec2ForCTC.from_pretrained(MID, target_lang='ava', ignore_mismatched_sizes=True).to(dev).eval()

PAL = 'Ӏ'
def fix(t):                       # палочка → Ӏ, лишние пробелы, первая буква заглавная
    t = re.sub(r'(?<=[А-Яа-яЁё])[IlІіӀӏ1]', PAL, t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t[:1].upper() + t[1:]

def norm(t):                      # для подсчёта ошибок: без регистра и пунктуации
    t = re.sub(r'(?<=[А-Яа-яЁё])[IlІіӀӏ1|]', PAL, t).lower().replace('ё', 'е').replace(PAL.lower(), PAL)
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\sӀ-]', ' ', t)).strip()

def lev(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

zin = zipfile.ZipFile(SRC)
rows = [l.split('|') for l in zin.read('gold.csv').decode('utf-8').splitlines() if l.strip()]
out_rows, E, N = [], 0, 0
for k, (path, gold, dur) in enumerate(rows):
    wav, sr = sf.read(io.BytesIO(zin.read(path)), dtype='float32')
    if wav.ndim > 1: wav = wav.mean(1)
    x = proc(wav, sampling_rate=sr, return_tensors='pt').input_values.to(dev)
    with torch.no_grad():
        ids = model(x).logits.argmax(-1)[0].cpu().tolist()
    draft = fix(proc.decode(ids))
    e = lev(norm(draft), norm(gold)); E += e; N += len(norm(gold))
    out_rows.append(f'{path}|{draft}|{dur}|{gold}')
    if k % 50 == 0: print(k, '/', len(rows), '|', draft[:60])

with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_STORED) as zo:
    for path, *_ in rows: zo.writestr(path, zin.read(path))
    zo.writestr('metadata.csv', '\n'.join(out_rows) + '\n')
print(f'\nГотово: {len(rows)} фрагментов, CER черновиков = {E/N:.1%}')
print('Скачай с Google Drive файл', OUT, '→ положи в папку авар\\site на компьютере')
